# TAPE Regression

In [1]:
#dataset = "fluorescence"
dataset = "stability"

### Model Registry

In [2]:
from protein_benchmark_models.models import ModelRegistry
print(ModelRegistry.list())

/Users/dylanelliott/workspace/protein-benchmark-models/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['ridge_regressor', 'mlp_regressor', 'cnn_regressor']


### Train Ridge Regressor

In [3]:
from protein_benchmark_models.data import OneHotSequenceDataset
import pandas as pd

# Train dataset
train_df = pd.read_csv(f".data/{dataset}/train.csv")
sequences = train_df["sequence"].tolist()
targets = train_df["target"].to_numpy()
max_seq_len = train_df["sequence"].map(lambda x: len(x)).max()
train_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Valid dataset
val_df = pd.read_csv(f".data/{dataset}/valid.csv")
sequences = val_df["sequence"].tolist()
targets = val_df["target"].to_numpy()
val_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Create model
model = ModelRegistry.get("ridge_regressor")(
    alpha=1.0
)

# Train, evaluate, and save
model.train(
    experiment_name=f"tape-{dataset}",
    train_data=train_dataset,
    val_data=val_dataset,
    model_path=f".models/tape_{dataset}_ridge_regressor",
    seed=42,
    tracking=True
)

[ridge_regressor] Train X: (53614, 1100)
[ridge_regressor] Train y: (53614,)
[ridge_regressor] Training model
[ridge_regressor] Valid RMSE: 0.5120
[ridge_regressor] Valid R2: 0.3903
[ridge_regressor] Valid SpearmanR: 0.5740
🏃 View run suave-slug-253 at: http://localhost:5000/#/experiments/3/runs/a5cc4b6be8484dd39a882fda9efb115b
🧪 View experiment at: http://localhost:5000/#/experiments/3


### Test Ridge Regressor

In [4]:
import numpy as np
from protein_benchmark_models.utils import evaluate

# Test dataset
test_df = pd.read_csv(f".data/{dataset}/test.csv")
sequences = test_df["sequence"].tolist()
targets = test_df["target"].to_numpy()
test_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Evaluate on test set
X = np.stack([test_dataset[i]["one_hots"].numpy().flatten() for i in range(len(test_dataset))])
y = test_dataset.targets.numpy()
metrics = evaluate(model, X, y)

print(f"Test RMSE: {metrics['rmse']:.04f}")
print(f"Test R2: {metrics['r2']:.04f}")
print(f"Test SpearmanR: {metrics['spearmanr']:.04f}")

Test RMSE: 0.7157
Test R2: -2.0662
Test SpearmanR: 0.4928


### Train MLP Regressor

In [5]:
from protein_benchmark_models.data import OneHotSequenceDataset
import pandas as pd

# Train dataset
train_df = pd.read_csv(f".data/{dataset}/train.csv")
sequences = train_df["sequence"].tolist()
targets = train_df["target"].to_numpy()
max_seq_len = train_df["sequence"].map(lambda x: len(x)).max()
train_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Valid dataset
val_df = pd.read_csv(f".data/{dataset}/valid.csv")
sequences = val_df["sequence"].tolist()
targets = val_df["target"].to_numpy()
val_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Create model
model = ModelRegistry.get("mlp_regressor")(
    layer_dims = [max_seq_len * len(train_dataset.vocab), 1024, 1],
    hidden_activation = "ReLU",
    output_activation = "Identity",
    use_bias = True,
    norm = "layer"
)

# Train, evaluate, and save
model.train(
    experiment_name=f"tape-{dataset}",
    train_data=train_dataset,
    val_data=val_dataset,
    model_path=f".models/tape_{dataset}_mlp_regressor",
    lr=1e-4,
    weight_decay = 0.01,
    max_epochs=5,
    batch_size=256,
    val_frequency=1,
    patience=25,
    save_model="best",
    seed=42,
    tracking=True
)

Epoch: 5/5 | train_loss: 0.3415 | val_loss: 0.4543 | best_val_loss: 0.4543: 100%|██████████| 5/5 [00:29<00:00,  5.97s/it]


[mlp_regressor] Valid RMSE: 0.4557
[mlp_regressor] Valid R2: 0.5169
[mlp_regressor] Valid SpearmanR: 0.6443
🏃 View run stately-dove-983 at: http://localhost:5000/#/experiments/3/runs/8622241bbc794da6b4186bc282e380e3
🧪 View experiment at: http://localhost:5000/#/experiments/3


### Test MLP Regressor

In [6]:
import numpy as np
from protein_benchmark_models.utils import evaluate

# Test dataset
test_df = pd.read_csv(f".data/{dataset}/test.csv")
sequences = test_df["sequence"].tolist()
targets = test_df["target"].to_numpy()
test_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Evaluate on test set
X = np.stack([test_dataset[i]["one_hots"].numpy().flatten() for i in range(len(test_dataset))])
y = test_dataset.targets.numpy()
metrics = evaluate(model, X, y)

print(f"Test RMSE: {metrics['rmse']:.04f}")
print(f"Test R2: {metrics['r2']:.04f}")
print(f"Test SpearmanR: {metrics['spearmanr']:.04f}")

Test RMSE: 0.4403
Test R2: -0.1607
Test SpearmanR: 0.5988


### Train CNN Regressor

In [7]:
from protein_benchmark_models.data import TokenizedSequenceDataset
import pandas as pd

# Train dataset
train_df = pd.read_csv(f".data/{dataset}/train.csv")
sequences = train_df["sequence"].tolist()
targets = train_df["target"].to_numpy()
max_seq_len = train_df["sequence"].map(lambda x: len(x)).max()
train_dataset = TokenizedSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Valid dataset
val_df = pd.read_csv(f".data/{dataset}/valid.csv")
sequences = val_df["sequence"].tolist()
targets = val_df["target"].to_numpy()
val_dataset = TokenizedSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Create model
model = ModelRegistry.get("cnn_regressor")(
    embed_dims=[len(train_dataset.vocab), 64],
    kernel_spec=[
        [3, 32, 1],
        [3, 64, 1],
        [3, 128, 2]
    ],
    seq_length=max_seq_len,
    output_dim=1,
    padding_idx=train_dataset.vocab['<PAD>'],
    hidden_activation="LeakyReLU",
    output_activation="Identity",
    use_bias=True,
    norm="layer"
)

# Train, evaluate, and save
model.train(
    experiment_name=f"tape-{dataset}",
    train_data=train_dataset,
    val_data=val_dataset,
    model_path=".models/tape_{dataset}_cnn_regressor",
    lr=1e-4,
    weight_decay = 0.01,
    max_epochs=5,
    batch_size=256,
    val_frequency=1,
    patience=25,
    save_model="best",
    seed=42,
    tracking=True
)

Epoch: 5/5 | train_loss: 0.4608 | val_loss: 0.4891 | best_val_loss: 0.4891: 100%|██████████| 5/5 [00:46<00:00,  9.26s/it]


[cnn_regressor] Valid RMSE: 0.4909
[cnn_regressor] Valid R2: 0.4394
[cnn_regressor] Valid SpearmanR: 0.5896
🏃 View run bald-donkey-575 at: http://localhost:5000/#/experiments/3/runs/eece164f38e34c5b831b57804b5eb4dc
🧪 View experiment at: http://localhost:5000/#/experiments/3


### Test CNN Regressor

In [8]:
import numpy as np
from protein_benchmark_models.utils import evaluate

# Test dataset
test_df = pd.read_csv(f".data/{dataset}/test.csv")
sequences = test_df["sequence"].tolist()
targets = test_df["target"].to_numpy()
test_dataset = TokenizedSequenceDataset(
    sequences=sequences,    
    targets=targets,
    seq_len=max_seq_len
)

# Evaluate on test set
X = np.stack([test_dataset[i]["tokens"].numpy() for i in range(len(test_dataset))])
y = test_dataset.targets.numpy()
metrics = evaluate(model, X, y)

print(f"Test RMSE: {metrics['rmse']:.04f}")
print(f"Test R2: {metrics['r2']:.04f}")
print(f"Test SpearmanR: {metrics['spearmanr']:.04f}")

Test RMSE: 0.6202
Test R2: -1.3029
Test SpearmanR: 0.5909
